# GRANAP for developers, 0: overview of the package

**For:** developers who want to read, check, change or couple the model. **You need:** the user tutorial
`../user/01_first_run.ipynb` (what the model does), Python, numpy. **Run time:** a few seconds.

This series:

| | notebook | what it covers |
|---|---|---|
| 0 | `00_overview.ipynb` | the package's modules, the original code, how the port is validated, where the equations are |
| 1 | `01_cross_sections_in_metafspm.ipynb` | metafspm's component formalism on `GranapAnatomy`, a plant model and its scenarios, several anatomies within a plant, the axial junctions, growth |
| 2 | `02_extend_the_component.ipynb` | add a variable and a process to the component, run and test it |

## 1. Where the model comes from

GRANAP is the Python translation of **GRANAR** (Heymans et al., 2020), extended to stems, leaves and needles. Its
original source is kept, frozen, as `openalea.granap.legacy`: the reference the port is compared with. The present
package generates every section into metafspm's multiscale plant graph (MPG), where its cells, walls, junctions and
hydraulic network are vertices and edges that other models (MECHA's water flows) read.

The docs (`doc/`) describe the component's declarations (`components.rst`), the rules and their equations, each with
the function computing it (`equations.rst`), the original code compared with the port and every departure, each with
its status (`legacy.rst`), and the timings (`performance.rst`).

## 2. The modules

The organ classes (roots, stems, leaves, needles) and their parameters are in `openalea.granap`; what is specific to
metafspm is in `openalea.granap.metafspm`. Each module's first docstring line:

In [1]:
import importlib
import pkgutil

import openalea.granap as package
import openalea.granap.metafspm as metafspm_package

for parent in (package, metafspm_package):
    for module in pkgutil.iter_modules(parent.__path__):
        if module.name == "legacy" or module.ispkg:
            continue
        name = f"{parent.__name__}.{module.name}"
        doc = (importlib.import_module(name).__doc__ or "").strip()
        print(f"{name.removeprefix('openalea.granap.'):24} {doc.splitlines()[0] if doc else ''}")

anatomy_writer           
cell_class               
cell_manager             
cellset_organ            Hybrid anatomy: real CellSet segmentation + a GRANAP-generated stele.
cellset_reader           CellSet XML ingestion — parse a digitized cross-section into polygons + tags.
distribution             Shape of a randomly drawn size (vessel, protoxylem, sieve-element diameter...).
generate_cell            Cell generator module for creating cells using Voronoi tessellation.
generate_cell_3d         3D cell generation via literal-copy 2D extrusion.
geometry_collection      Geometry processor module for handling polygon operations.
input_data               
layer_class              Layer module for plant anatomy representation.
layer_manager            Layer manager module for handling collections of tissue layers.
leaf_class               Leaf cross-section anatomy.
math_functions           Normalized shape functions and a rescale wrapper for GRANAP gradients.
mesh_utils_3d            Small

The ones to know first:

- `input_data.py` holds the parameters (`OrganInputData` and its presets, one pydantic model per section of the
  anatomy);
- `root_class.py`, `stem_class.py`, `leaf_class.py`, `needle_class.py` generate the organs' sections;
- `metafspm/component.py` holds the component, `GranapAnatomy`. Tutorial 1 of this series reads it;
- `metafspm/architectures.py` gives stand-in plant structures (`RootAxis`, `Needle`) for examples and tutorials;
- `simulation.py` is the users' entry point: `simulate`, the summary and the tables.

## 3. How the port is validated

The tests, in `test/`, run with `pytest test` from the repository's root. The organs' own tests are at its top;
`test/component/` tests the metafspm component, `test/comparison/` the port against the original code. Each file
checks one aspect:

In [2]:
import ast
from pathlib import Path

for folder in ("test", "test/component", "test/comparison"):
    for path in sorted(Path("../../..", folder).glob("test_*.py")):
        doc = ast.get_docstring(ast.parse(path.read_text())) or ""
        print(f"{folder + '/' + path.name:52} {' '.join(doc.split())[:110]}")

test/test_cells_3d.py                                Tests for Organ.generate_cells_3d() / generate_cell_3d.py: literal-copy 2D-cell extrusion, no 3D Voronoi. The 
test/test_cellset_organ.py                           CellSet ingestion + the GRANAP-stele graft (``CellSetOrgan``). The fixture is a real CellSet digitization of a
test/test_cgroups.py                                 Every cell type GRANAP's presets generate has its MECHA cgroup (anatomy_writer.CGROUP_MAP): none falls back to
test/test_distribution.py                            Size distributions (openalea.granap.distribution) and their wiring into the params. The default (no distributi
test/test_growth_modes.py                            Structure tests for the growth cases the golden suite does not pin. ``test_vascular_regression`` covers the mo
test/test_leaf_slab.py                               Checks for the leaf cross-section (LeafAnatomy: monocot slab + dicot).
test/test_medullar_rays.py                           Tes

GRANAP's sections are random (the cells' positions and sizes are drawn), with a seed: a seed always gives the same
section. `test/test_vascular_regression.py` pins the exact census of reference sections (`test/golden`). It depends
on the geometry libraries' builds: `conda/environment.yaml` pins them. The comparison with the original code is in
`test/comparison` and `example/comparison`, and every difference is registered in the legacy page.

To rerun one file while working: `pytest test/test_simulation.py -q`.

## 4. What to read next

- `01_cross_sections_in_metafspm.ipynb` reads the component's code through metafspm's component formalism. It shows
  a plant model, its scenarios, the classes of anatomies within a plant and how sections are joined.
- `02_extend_the_component.ipynb` adds a variable and a process.
- MECHA's tutorials (MECHA-dev) compute the water flows on these sections.
- metafspm's own tutorial (`examples/soil_plant_atmosphere/tutorial_fspm2026_general_usage.ipynb` in the metafspm
  repository) builds components and couples them step by step. Its user guide, `docs/user.md`, describes every
  part of the framework.